In [1]:
import pandas as pd
import numpy as np
import datetime as dt

# dtw-python 
from dtw import dtw

import copy

import plotly.express as px
import matplotlib.pyplot as plt
%matplotlib inline

Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



In [3]:
print(dtw)

<function dtw at 0x000001FE9AE40AE0>


In [4]:
data = pd.read_csv("../data/preprocessed_data.csv")
data.head()

,Unnamed: 0,timestamp,load_Rotary_C5_temp,execution,program_name,nsequence,cr,index
0,46,2024-03-11 14:01:19.408233,0.014493,0,O0005(5303-005-C),N10,CR-1,0
1,49,2024-03-11 14:01:29.432814,0.014493,0,O0005(5303-005-C),N10,CR-1,1
2,63,2024-03-11 14:01:37.451878,0.000000,0,O0005(5303-005-C),N10,CR-1,2
3,64,2024-03-11 14:01:44.468672,0.000000,0,O0005(5303-005-C),N10,CR-1,3
4,78,2024-03-11 14:01:52.489886,0.000000,0,O0005(5303-005-C),N10,CR-1,4


In [ ]:
def euclidean_distance(x, y):
    return abs(x - y)  

def dtw_cost_alignment(df):
    size = len(df['cr'].value_counts())
    dtw_costs = [[0 for _ in range(size)] for _ in range(size)]
    for i in range(size):
        for j in range(i, size):
            df_load_template = df.loc[df['cr'] == f'CR-{i+1}', 'load_Rotary_C5_temp'].values
            df_load_template = df_load_template[~np.isnan(df_load_template)]

            df_load_query = df.loc[df['cr'] == f'CR-{j+1}', 'load_Rotary_C5_temp'].values
            df_load_query = df_load_query[~np.isnan(df_load_query)]

            dtw_obj = dtw(df_load_template, df_load_query,dist_method=euclidean_distance)
            # print("test: ", dtw_cost[0])
            dtw_cost = dtw_obj.distance()
            cost = round(dtw_cost[0],5)
            print(f'CR: {i+1}-{j+1} COST: {cost}')
            dtw_costs[i][j] = cost
            dtw_costs[j][i] = cost
    return dtw_costs


In [12]:
n10_dtw_cost = dtw_cost_alignment(data) 

ValueError: setting an array element with a sequence.

In [7]:
def cost_per_time_stamp(df, cost):
    costs_size = len(df['cr'].value_counts())
    new_costs = [[0 for _ in range(costs_size)] for _ in range(costs_size)]
    for i in range(costs_size):
        for j in range(i, costs_size):
            df_load_query = df.loc[df['cr'] == f'CR-{j+1}', 'load_Rotary_C5_temp'].values
            df_load_query = df_load_query[~np.isnan(df_load_query)]

            size = len(df_load_query)
            new_cost = round(cost[i][j]/size, 5)
            
            new_costs[i][j] = new_cost
            new_costs[j][i] = new_cost
            
    return new_costs
    

In [ ]:
def get_crs_within_threshold(cost, threshold):
    size = len(cost)
    arr = copy.deepcopy(cost)
    crs_in_threshold = [[set([i + 1, j + 1]) for j in range(size)] for i in range(size)]
#     print(crs_in_threshold)
    max_index_i = 0
    max_index_j = 0
    for gap in range(2, size):
        for i, j in zip(range(size - gap), range(gap, size)): 
            cr_i = i + 1,
            cr_j = j + 1,
            min_cost = min(arr[i][j], arr[i][j - 1], arr[i + 1][j])
            if arr[i][j] <= threshold:
                if arr[i][j - 1] <= threshold and arr[i + 1][j] <= threshold:
#                     print(i, j, crs_in_threshold[i][j], crs_in_threshold[i][j - 1], crs_in_threshold[i + 1][j])
                    crs_in_threshold[i][j] = crs_in_threshold[i][j].union(crs_in_threshold[i][j - 1])
                    crs_in_threshold[i][j] = crs_in_threshold[i][j].union(crs_in_threshold[i + 1][j])
            else:
                min_cost = min(arr[i][j - 1], arr[i + 1][j])
                if arr[i][j - 1] == min_cost and arr[i][j - 1] <= threshold:
#                     arr[i][j] = min_cost
                    crs_in_threshold[i][j] = crs_in_threshold[i][j - 1]
                elif arr[i + 1][j] == min_cost and arr[i + 1][j] <= threshold:
#                     arr[i][j] = min_cost
                    crs_in_threshold[i][j] = crs_in_threshold[i + 1 ][j]
                else:
                    crs_in_threshold[i][j] = set([])
            if len(crs_in_threshold[i][j]) > len(crs_in_threshold[max_index_i][max_index_j]):
                max_index_i = i
                max_index_j = j
    return crs_in_threshold[max_index_i][max_index_j]  
    

In [ ]:
n10_dtw_cost_per_timestamp = cost_per_time_stamp(data, n10_dtw_cost)

In [ ]:
def display_crs(costs, df, threshold):
    cr_set = get_crs_within_threshold(costs, threshold)
    print('num CRs: ', len(cr_set))
    
    df['cr_numeric'] = df['cr'].str.extract(r'(\d+)')
    df['cr_numeric'] = pd.to_numeric(df['cr_numeric'])
    filtered_df = df[df['cr_numeric'].isin(cr_set)]
    filtered_df = filtered_df.reset_index(drop=True)
    # filtered_df['index'] = filtered_df.groupby(['nsequence', 'cr_numeric']).cumcount()
    filtered_df['index'] = filtered_df.groupby(['cr_numeric']).cumcount()
#     print(filtered_df['cr'].value_counts())
    
    fig = px.line(filtered_df, x="index", y="load_Rotary_C5", color='cr')
    fig.update_layout(title_text="Load Vs Index")
    fig.show()
    return cr_set, filtered_df

In [ ]:
# Nsequence: N10
# Threshold: 0.003
df10_cr_set, dfN10_load_filtered_cr = display_crs(n10_dtw_cost_per_timestamp, data, 0.03)